# ChatGPT code

In [1]:
import numpy as np


def positive_normalize(x):
    """
    正向指标：越大越好
    """
    x = np.array(x, dtype=float)

    min_value = x.min()
    max_value = x.max()

    if max_value == min_value:
        return np.ones_like(x)

    return (x - min_value) / (max_value - min_value)


def negative_normalize(x):
    """
    负向指标：越小越好
    """
    x = np.array(x, dtype=float)

    min_value = x.min()
    max_value = x.max()

    if max_value == min_value:
        return np.ones_like(x)

    return (max_value - x) / (max_value - min_value)

In [2]:
import numpy as np


X = np.array([
    [100, 20, 80],
    [150, 50, 50],
    [200, 90, 20]
], dtype=float)


# 第 1、2 列是正向指标
X_pos = X[:, [0, 1]]

# 第 3 列是负向指标
X_neg = X[:, [2]]


def positive_normalize(x):
    min_value = x.min(axis=0)
    max_value = x.max(axis=0)

    denominator = max_value - min_value

    return (x - min_value) / denominator


def negative_normalize(x):
    min_value = x.min(axis=0)
    max_value = x.max(axis=0)

    denominator = max_value - min_value

    return (max_value - x) / denominator


Z_pos = positive_normalize(X_pos)
Z_neg = negative_normalize(X_neg)

Z = np.hstack([Z_pos, Z_neg])

print(Z)

[[0.         0.         0.        ]
 [0.5        0.42857143 0.5       ]
 [1.         1.         1.        ]]


In [3]:
def entropy_weight_method(X, positive_columns):
    """
    熵权法

    X:
        原始数据矩阵，shape = (m, n)

    positive_columns:
        正向指标列的下标，例如 [0, 1]
    """

    X = np.array(X, dtype=float)

    m, n = X.shape

    # -------------------------
    # 1. 标准化
    # -------------------------

    Z = np.zeros_like(X)

    for j in range(n):

        column = X[:, j]

        max_value = column.max()
        min_value = column.min()

        # 如果全部一样
        if max_value == min_value:
            Z[:, j] = 1
            continue

        if j in positive_columns:

            # 正向指标
            Z[:, j] = (
                column - min_value
            ) / (
                max_value - min_value
            )

        else:

            # 负向指标
            Z[:, j] = (
                max_value - column
            ) / (
                max_value - min_value
            )

    # -------------------------
    # 2. 计算比重 P
    # -------------------------

    column_sum = Z.sum(axis=0)

    # 防止除以 0
    P = np.zeros_like(Z)

    for j in range(n):

        if column_sum[j] == 0:
            P[:, j] = 1 / m
        else:
            P[:, j] = Z[:, j] / column_sum[j]

    # -------------------------
    # 3. 计算熵
    # -------------------------

    eps = 1e-12

    E = np.zeros(n)

    for j in range(n):

        p = P[:, j]

        E[j] = (
            -np.sum(
                p * np.log(p + eps)
            )
            / np.log(m)
        )

    # -------------------------
    # 4. 信息效用值
    # -------------------------

    D = 1 - E

    # -------------------------
    # 5. 熵权
    # -------------------------

    W = D / D.sum()

    return Z, P, E, D, W

In [4]:
X = np.array([
    [100, 20, 80],
    [150, 50, 50],
    [200, 90, 20]
])

Z, P, E, D, W = entropy_weight_method(
    X,
    positive_columns=[0, 1]
)

print("标准化矩阵：")
print(Z)

print("\n比重矩阵：")
print(P)

print("\n熵：")
print(E)

print("\n信息效用值：")
print(D)

print("\n最终权重：")
print(W)

标准化矩阵：
[[0.         0.         0.        ]
 [0.5        0.42857143 0.5       ]
 [1.         1.         1.        ]]

比重矩阵：
[[0.         0.         0.        ]
 [0.33333333 0.3        0.33333333]
 [0.66666667 0.7        0.66666667]]

熵：
[0.57938016 0.55603265 0.57938016]

信息效用值：
[0.42061984 0.44396735 0.42061984]

最终权重：
[0.32727788 0.34544423 0.32727788]


# Gemini code

In [5]:
import numpy as np
import pandas as pd

def entropy_weight(df, pos_cols, neg_cols):
    """
    计算熵权法权重
    :param df: 包含数据的 DataFrame
    :param pos_cols: 正向指标列名列表
    :param neg_cols: 负向指标列名列表
    :return: 各指标权重
    """
    # 1. 提取需要计算的指标数据
    cols = pos_cols + neg_cols
    data = df[cols].astype(float).copy()
    n, m = data.shape
    
    # 2. 数据标准化
    for col in pos_cols:
        data[col] = (data[col] - data[col].min()) / (data[col].max() - data[col].min())
        
    for col in neg_cols:
        data[col] = (data[col].max() - data[col]) / (data[col].max() - data[col].min())
        
    # 平移极小值，避免出现 log(0) 的情况
    data = data + 1e-5
    
    # 3. 计算比重 P
    P = data / data.sum(axis=0)
    
    # 4. 计算信息熵 E
    E = - (1 / np.log(n)) * (P * np.log(P)).sum(axis=0)
    
    # 5. 计算效用值 D 和 最终权重 W
    D = 1 - E
    W = D / D.sum()
    
    return W

# ================= 运行示例 =================
# 构建数据
data_dict = {
    '手机型号': ['A', 'B', 'C', 'D'],
    '价格': [4000, 5000, 3500, 6000],
    '续航': [10, 12, 8, 14],
    '拍照': [85, 90, 80, 95]
}
df = pd.DataFrame(data_dict)

# 定义正向和负向指标
positive_indicators = ['续航', '拍照']
negative_indicators = ['价格']

# 计算权重
weights = entropy_weight(df, positive_indicators, negative_indicators)

print("各项指标的熵权法权重为：")
for index, value in weights.items():
    print(f"{index}: {value:.4f}")

# 计算综合得分 (标准化后的数据乘以权重)
# 为方便展示，这里重新进行一次不加 1e-5 的严格标准化来算最终得分
norm_data = df.copy()
norm_data['价格'] = (df['价格'].max() - df['价格']) / (df['价格'].max() - df['价格'].min())
norm_data['续航'] = (df['续航'] - df['续航'].min()) / (df['续航'].max() - df['续航'].min())
norm_data['拍照'] = (df['拍照'] - df['拍照'].min()) / (df['拍照'].max() - df['拍照'].min())

df['综合得分'] = norm_data[positive_indicators + negative_indicators].dot(weights)
print("\n各手机的最终综合得分为：")
print(df[['手机型号', '综合得分']].sort_values(by='综合得分', ascending=False).to_string(index=False))

各项指标的熵权法权重为：
续航: 0.3408
拍照: 0.3408
价格: 0.3183

各手机的最终综合得分为：
手机型号     综合得分
   D 0.681694
   B 0.581785
   A 0.481876
   C 0.318306
